# RaTEScore metrics

**RaTEScore** (Radiological Report Text Evaluation) was explicitly designed to fix **RadGraph**'s flaws.

To install RaTEScore metrics:

    pip uninstall transformers  # uninstall old "transformers<5.0.0" version for RadGraph
    pip install transformers  # install new version 5.12.1
    pip install RaTEScore


In [1]:
GPU = 3  # GPU number. After change, restart the kernel

%set_env CUDA_VISIBLE_DEVICES={GPU}

import os
import torch

if torch.cuda.is_available():  # make sure GPU is available
    num = torch.cuda.device_count()
    print(f"GPU count: {num}")
    for i in range(num):
        print(f"GPU {i} name: {torch.cuda.get_device_name(i)}")
else:
    print("GPU is not available")

# Disable Hugging Face tokenizer serialization forks warning
os.environ["TOKENIZERS_PARALLELISM"] = "false"

env: CUDA_VISIBLE_DEVICES=3
GPU count: 1
GPU 0 name: Tesla V100-PCIE-16GB


In [2]:
import os
import sys
import torch
import pandas as pd

from loguru import logger
from tqdm.notebook import tqdm
from RaTEScore import RaTEScore  # import the official RaTEScore package


BATCH_SIZE = 512


# Remove the default loguru handler and reconfigure it to only handle important messages
logger.remove()
logger.add(sys.stderr, level="INFO") # blocks everything below INFO (including DEBUG)


def calculate_ratescore(predicted_list, actual_list, show=True):
    """
    Computes the official RaTEScore metric for radiology reports.
    Perfectly compatible with Python 3.13+, Torch 2.6.0, and modern Transformers.
    """
    if show:
        print("--- Running RaTEScore Evaluation ---")

    # --- SUPPRESS INTERNAL HUB LOGS START ---
    old_stdout = sys.stdout
    sys.stdout = open(os.devnull, "w")

    try:
        # Initialize the evaluator
        # affinity_matrix='short' is ideal for quick semantic mappings.
        # Use affinity_matrix='long' if your reports contain massive paragraphs.
        # batch_size=512 packs multiple reports into a single GPU operations grid, 
        # instantly utilizing available VRAM (1.7GB -> ~12GB) and filling up CPU threads.
        evaluator = RaTEScore(affinity_matrix="short", batch_size=BATCH_SIZE)
    finally:
        sys.stdout.close()
        sys.stdout = old_stdout
    # --- SUPPRESS INTERNAL HUB LOGS END ---

    # Compute the scores
    # RaTEScore processes the lists globally or automatically handles internal batching
    scores = evaluator.compute_score(predicted_list, actual_list)

    # Calculate the global mean score for the column
    mean_ratescore = sum(scores) / len(scores)

    if show:
        print(f"Mean RaTEScore: {mean_ratescore:.4f}")

    return mean_ratescore


def calculate_all_metrics(csv_file):
    """ Reads your results CSV and evaluates columns using RaTEScore """
    df = pd.read_csv(csv_file)
    actual_reports = df["actual_text"].tolist()
    columns_list = df.columns.tolist()

    print("=== Starting RaTEScore Evaluation Pipeline ===")

    # Iterate through your model generation columns (skipping filename and ground truth)
    for col in tqdm(columns_list[2:], desc="Overall Progress"):
        predicted_reports = df[col].tolist()

        # Calculate score with silent initialization
        score = calculate_ratescore(predicted_reports, actual_reports, show=False)
        print(f"\t{col}: {score:.3f}")


if __name__ == "__main__":
    # Run over your existing test file
    calculate_all_metrics("../test_all.csv")

=== Starting RaTEScore Evaluation Pipeline ===


Overall Progress:   0%|          | 0/17 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

	test_base_qwen: 0.308


...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

	best_model_lstm_5_layers_resnet_unfreeze_top_p=0.4: 0.555


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

	best_model_lstm_5_layers_resnet_unfreeze_top_p=0.5: 0.554
